In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv(r"C:\Users\A-FF-L1-D07\Desktop\03_Plants_النباتات-20260830T162732Z-1-001\03_Plants_النباتات\03_manual_requirements_sources.csv")

In [3]:
df["species"] = (
    df["species"]
    .str.strip()                            # remove leading/trailing whitespace
    .str.replace(r"\s+", " ", regex=True))   # collapse multiple internal spaces into one

In [4]:
def split_field_value(field: str, value: str):
    """
    Takes one raw (field, value) pair and returns a list of clean
    (parameter_name, parameter_value) pairs — one row becomes one or two rows.
    """
    field = field.strip()
    value = value.strip()

    if " / " not in field:
        # simple case: one field, one value -> one row stays one row
        return [(field, value)]

    # composite case: split field name (same logic as before)
    left, right_suffix = [p.strip() for p in field.split(" / ", 1)]
    prefix = left[: -len("min")] if left.endswith("min") else left
    right_name = prefix + right_suffix.split("_")[-1]

    # split the matching value the same way: "10 / 25" -> "10", "25"
    if " / " in value:
        v_left, v_right = [v.strip() for v in value.split(" / ", 1)]
        return [(left, v_left), (right_name, v_right)]
    else:
        return [(left, value), (right_name, None)]  # fallback if value isn't split the same way


# Build the new tidy rows
tidy_rows = []
for _, row in df.iterrows():
    for parameter, val in split_field_value(row["field"], row["value"]):
        tidy_rows.append({
            "species": row["species"],
            "parameter": parameter,   # <-- new column, replaces "field"
            "value": val,             # <-- same column name, but now always one value
        })

tidy_df = pd.DataFrame(tidy_rows)

In [5]:
tidy_df = pd.DataFrame(tidy_rows)

# ---- Show the result ----
print("Original shape:", df.shape)      # rows before splitting
print("New shape:", tidy_df.shape)      # rows after splitting (should be more)
print()
print(tidy_df.head(20).to_string())     # show first 20 rows fully

# Example: show only the rows that came from one specific species,
# to see clearly how one row became two
print()
print(tidy_df[tidy_df["species"] == "Ammi visnaga"].to_string())

Original shape: (55, 6)
New shape: (82, 3)

               species              parameter      value
0         Ammi visnaga  temperature_c_opt_min         10
1         Ammi visnaga  temperature_c_opt_max         25
2         Ammi visnaga  temperature_c_abs_min          5
3         Ammi visnaga  temperature_c_abs_max         32
4         Ammi visnaga        soil_ph_abs_min        6.8
5         Ammi visnaga        soil_ph_abs_max        8.3
6         Ammi visnaga        soil_ph_opt_min        7.0
7         Ammi visnaga        soil_ph_opt_max        8.0
8         Ammi visnaga        crop_cycle_days    210-240
9         Ammi visnaga               rainfall  NOT FOUND
10          Ammi majus  temperature_c_opt_min         20
11          Ammi majus  temperature_c_opt_max         22
12          Ammi majus  temperature_c_abs_min          8
13          Ammi majus  temperature_c_abs_max         35
14          Ammi majus        soil_ph_abs_min        6.0
15          Ammi majus        soil_ph_abs_ma

In [6]:
def expand_special_cases(parameter: str, value: str):
    """
    Handles values that pack more than one number in a single cell,
    where the 'parameter' name itself doesn't tell us they're separate.
    Returns a list of (parameter, value) pairs.
    """
    value = value.strip()

    # Case 1: rainfall_mm = "opt 700-1000, abs 300-1300"
    if parameter == "rainfall_mm" and "opt" in value and "abs" in value:
        m_opt = re.search(r"opt\s*([\d.]+)-([\d.]+)", value)
        m_abs = re.search(r"abs\s*([\d.]+)-([\d.]+)", value)
        result = []
        if m_opt:
            result += [("rainfall_mm_opt_min", m_opt.group(1)), ("rainfall_mm_opt_max", m_opt.group(2))]
        if m_abs:
            result += [("rainfall_mm_abs_min", m_abs.group(1)), ("rainfall_mm_abs_max", m_abs.group(2))]
        return result

    # Case 2: crop_cycle_days = "56 (leaf) / 140 (seed)"
    if parameter == "crop_cycle_days" and "(" in value:
        parts = re.findall(r"([\d.]+)\s*\(([^)]+)\)", value)
        if parts:
            return [(f"crop_cycle_days_{label.strip()}", num) for num, label in parts]

    # default: no special case, keep as is
    return [(parameter, value)]


# --- Step B: general-purpose numeric parser for everything else ---

def parse_value(raw: str):
    raw = str(raw).strip()

    # "NOT FOUND" -> missing data, not an error
    if raw.upper() == "NOT FOUND" or raw == "" or raw == "None":
        return pd.Series({"value_min": np.nan, "value_max": np.nan,
                           "value_text": np.nan, "unit": np.nan, "is_missing": True})

    approx = raw.startswith("~")
    raw_clean = raw.lstrip("~").strip()

    # detect a unit like "dS/m"
    unit = "dS/m" if re.search(r"dS/m", raw_clean) else np.nan
    text_wo_unit = re.sub(r"dS/m", "", raw_clean)

    # pull out all numbers in the string (handles decimals)
    nums = re.findall(r"\d+\.?\d*", raw_clean)

    if not nums:
        # pure text, e.g. "good", "check required"
        return pd.Series({"value_min": np.nan, "value_max": np.nan,
                           "value_text": raw_clean, "unit": np.nan, "is_missing": False})

    if len(nums) == 1:
        v = float(nums[0])
        descriptive_text = re.sub(r"[\d.]+", "", text_wo_unit).strip(" -") or np.nan
        return pd.Series({"value_min": v, "value_max": v,
                           "value_text": descriptive_text, "unit": unit, "is_missing": False})

    # two numbers -> a range: first = min, second = max
    v_min, v_max = float(nums[0]), float(nums[1])
    descriptive_text = re.sub(r"[\d.]+", "", text_wo_unit).strip(" -,") or np.nan
    return pd.Series({"value_min": v_min, "value_max": v_max,
                       "value_text": descriptive_text, "unit": unit, "is_missing": False})


# --- Step C: apply both steps on top of the tidy_df from before ---

expanded_rows = []
for _, row in tidy_df.iterrows():
    for parameter, val in expand_special_cases(row["parameter"], str(row["value"])):
        expanded_rows.append({"species": row["species"], "parameter": parameter, "value_raw": val})

expanded_df = pd.DataFrame(expanded_rows)

parsed = expanded_df["value_raw"].apply(parse_value)
final_df = pd.concat([expanded_df, parsed], axis=1)

print("Shape after expanding special cases:", final_df.shape)
print(final_df.head(15).to_string())
print()
print("Rows flagged as missing:", final_df["is_missing"].sum())

Shape after expanding special cases: (89, 8)
         species              parameter  value_raw  value_min  value_max value_text unit  is_missing
0   Ammi visnaga  temperature_c_opt_min         10       10.0       10.0        NaN  NaN       False
1   Ammi visnaga  temperature_c_opt_max         25       25.0       25.0        NaN  NaN       False
2   Ammi visnaga  temperature_c_abs_min          5        5.0        5.0        NaN  NaN       False
3   Ammi visnaga  temperature_c_abs_max         32       32.0       32.0        NaN  NaN       False
4   Ammi visnaga        soil_ph_abs_min        6.8        6.8        6.8        NaN  NaN       False
5   Ammi visnaga        soil_ph_abs_max        8.3        8.3        8.3        NaN  NaN       False
6   Ammi visnaga        soil_ph_opt_min        7.0        7.0        7.0        NaN  NaN       False
7   Ammi visnaga        soil_ph_opt_max        8.0        8.0        8.0        NaN  NaN       False
8   Ammi visnaga        crop_cycle_days    210

In [7]:
# Step 1: normalize text (lowercase + strip)
df["evidence_grade"] = df["evidence_grade"].str.strip().str.lower()

# Step 2: replace the ambiguous "-" with an explicit, readable label
df["evidence_grade"] = df["evidence_grade"].replace({"-": "not_applicable"})

# Step 3 (optional but useful): rank evidence reliability numerically
grade_rank = {
    "measured": 1,
    "measured (proxy)": 2,
    "literature": 3,
    "database": 4,
    "habitat": 5,
    "derived": 6,
    "inferred": 7,
    "not_applicable": np.nan,
}
df["evidence_rank"] = df["evidence_grade"].map(grade_rank)

print(df["evidence_grade"].value_counts())

evidence_grade
database            18
measured            13
derived              9
not_applicable       8
habitat              4
literature           1
inferred             1
measured (proxy)     1
Name: count, dtype: int64


In [8]:
# Step 1: basic text cleanup
df["note"] = df["note"].str.strip()
# Replace missing notes with an explicit text label instead of NaN
df["note"] = df["note"].replace({"": "Not specified"})
df["note"] = df["note"].fillna("Not specified")
# Step 3: flag any row where a source conflict was explicitly disclosed
df["has_conflict"] = df["note"].str.contains("CONFLICT", case=False, na=False)

print(df["has_conflict"].value_counts())


has_conflict
False    53
True      2
Name: count, dtype: int64


In [9]:
print(df[df["has_conflict"]][["species", "field", "note"]].to_string())

                  species            field                                                                                                                                                                                                                                                     note
19  Matricaria chamomilla  soil_ph_abs_max                      CONFLICT DISCLOSED: the database ceiling is 7.0, but Ain Shams University grew a successful crop at Shoubra El-Kheima on soil of pH 7.77 and 8.27. Egyptian field evidence used; database figure recorded as a documented conflict.
26              Aloe vera      rainfall_mm  CONFLICT DISCLOSED: Useful Tropical Plants gives an optimum of 1900-3000 mm, which contradicts Aloe's desert-succulent physiology. The Odisha SOP figure of 1000-1800 mm with irrigation is used as the more defensible input for an Egyptian scenario.


In [10]:
# Step 1: basic text cleanup
df["source_url"] = df["source_url"].str.strip()

# Step 2: convert empty strings to a proper missing value
df["source_url"] = df["source_url"].replace({"": np.nan})

# Step 3: check that non-missing URLs actually look like valid URLs
df["source_url_valid"] = df["source_url"].apply(
    lambda url: bool(re.match(r"^https?://", url)) if pd.notna(url) else np.nan
)

print(df["source_url_valid"].value_counts(dropna=False))
print(df[df["source_url"].isna()][["species", "field", "note"]].to_string())

source_url_valid
True    53
NaN      2
Name: count, dtype: int64
         species     field                                                                                                     note
5   Ammi visnaga  rainfall  No published rainfall requirement. All Egyptian production is irrigated. Field left blank deliberately.
10    Ammi majus  rainfall                                                       No published requirement. Irrigated crop in Egypt.


In [1]:
import pandas as pd
import numpy as np
import re
df = pd.read_csv(r"C:\Users\A-FF-L1-D07\Desktop\03_Plants_النباتات-20260830T162732Z-1-001\03_Plants_النباتات\03_manual_requirements_sources.csv")

df["species"] = (
    df["species"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True))
def split_field_value(field, value):
    """
    Returns a list of (parameter_name, raw_value) tuples for one row.
    Covers every case that actually occurs in this file.
    """
    field = field.strip()
    value = value.strip()

    # Case A: composite field name "X_min / X_max" -> must be split into two fields
    if " / " in field:
        left, right_suffix = [p.strip() for p in field.split(" / ", 1)]
        # left = "temperature_c_opt_min" -> strip trailing "min" to get the shared prefix
        prefix = left[: -len("min")] if left.endswith("min") else left
        right_name = prefix + right_suffix.split("_")[-1]  # appends "max"
        # the matching value is usually "10 / 25"
        if " / " in value:
            v_left, v_right = [v.strip() for v in value.split(" / ", 1)]
            return [(left, v_left), (right_name, v_right)]
        else:
            # fallback: if the value isn't split the same way, keep the full value on the first name
            return [(left, value), (right_name, np.nan)]

    # Case B: rainfall_mm holds both opt & abs values packed into one text cell
    # Example: "opt 700-1000, abs 300-1300"
    if field == "rainfall_mm" and "opt" in value and "abs" in value:
        m_opt = re.search(r"opt\s*([\d.]+)-([\d.]+)", value)
        m_abs = re.search(r"abs\s*([\d.]+)-([\d.]+)", value)
        out = []
        if m_opt:
            out += [("rainfall_mm_opt_min", m_opt.group(1)), ("rainfall_mm_opt_max", m_opt.group(2))]
        if m_abs:
            out += [("rainfall_mm_abs_min", m_abs.group(1)), ("rainfall_mm_abs_max", m_abs.group(2))]
        return out if out else [(field, value)]

    # Case C: crop_cycle_days sometimes comes with two different products "56 (leaf) / 140 (seed)"
    if field == "crop_cycle_days" and "(" in value and "/" in value:
        parts = re.findall(r"([\d.]+)\s*\(([^)]+)\)", value)
        return [(f"crop_cycle_days_{label.strip()}", num) for num, label in parts]

    # Case D: everything else — a single field with a single value (number / range / descriptive text)
    return [(field, value)]


tidy_rows = []
for _, row in df.iterrows():
    pairs = split_field_value(row["field"], row["value"])
    for parameter, raw_val in pairs:
        tidy_rows.append({
            "species": row["species"],
            "parameter": parameter,
            "value_raw": raw_val,
            "evidence_grade": row["evidence_grade"],
            "note": row["note"],
            "source_url": row["source_url"],
        })

tidy = pd.DataFrame(tidy_rows)

# ---------------------------------------------------------------
# 4) Converting value_raw into real numbers (numeric) + unit + "missing value" flag
# ---------------------------------------------------------------
# The numeric fields come in different shapes:
#   "210-240"                    -> a range
#   "10"                         -> a single number
#   "~6 dS/m practical ceiling"  -> an approximate number + unit + description
#   "9-30 dS/m reported range"   -> a range + unit + description
#   "NOT FOUND"                  -> data not available -> NaN + is_missing=True flag

def parse_value(raw):
    raw = str(raw).strip()

    if raw.upper() == "NOT FOUND" or raw == "":
        return pd.Series({"value_min": np.nan, "value_max": np.nan,
                           "value_text": np.nan, "unit": np.nan, "is_missing": True})

    # strip the "~" (approximation) symbol
    raw_clean = raw.lstrip("~").strip()

    # detect a unit like "dS/m" if present
    unit_match = re.search(r"dS/m", raw_clean)
    unit = "dS/m" if unit_match else np.nan

    # extract any numbers found in the text (supports decimals and ranges separated by "-")
    nums = re.findall(r"\d+\.?\d*", raw_clean)

    if not nums:
        # no numbers at all -> purely descriptive text (e.g. "good", "marginal", "check required")
        return pd.Series({"value_min": np.nan, "value_max": np.nan,
                           "value_text": raw_clean, "unit": np.nan, "is_missing": False})

    # remove the unit from the text before extracting any extra description
    # (so it doesn't get duplicated inside value_text)
    text_wo_unit = re.sub(r"dS/m", "", raw_clean)

    if len(nums) == 1:
        v = float(nums[0])
        descriptive_text = re.sub(r"[\d.]+", "", text_wo_unit).strip(" -")
        descriptive_text = descriptive_text if descriptive_text else np.nan
        return pd.Series({"value_min": v, "value_max": v,
                           "value_text": descriptive_text, "unit": unit, "is_missing": False})

    # two numbers found -> a range: first number = min, second number = max
    v_min, v_max = float(nums[0]), float(nums[1])
    descriptive_text = re.sub(r"[\d.]+", "", text_wo_unit).strip(" -,")
    descriptive_text = descriptive_text if descriptive_text else np.nan
    return pd.Series({"value_min": v_min, "value_max": v_max,
                       "value_text": descriptive_text, "unit": unit, "is_missing": False})


parsed = tidy["value_raw"].apply(parse_value)
tidy = pd.concat([tidy, parsed], axis=1)

# ---------------------------------------------------------------
# 5) Column: evidence_grade — standardizing categories (categorical cleaning)
# ---------------------------------------------------------------
# Existing values: derived, database, measured, literature, inferred, habitat,
#                  "measured (proxy)", and "-" (not applicable: either missing data
#                  or a descriptive field like EGYPT SUITABILITY)
tidy["evidence_grade"] = tidy["evidence_grade"].str.strip().str.lower()
tidy["evidence_grade"] = tidy["evidence_grade"].replace({"-": "not_applicable"})

# Ordinal reliability ranking of the evidence — useful for a later confidence analysis
grade_rank = {
    "measured": 1, "measured (proxy)": 2, "literature": 3,
    "database": 4, "habitat": 5, "derived": 6,
    "inferred": 7, "not_applicable": np.nan,
}
tidy["evidence_rank"] = tidy["evidence_grade"].map(grade_rank)

# ---------------------------------------------------------------
# 6) Column: note (free text)
# ---------------------------------------------------------------
tidy["note"] = tidy["note"].str.strip()
tidy["note"] = tidy["note"].replace({"": "Not specified"})
tidy["note"] = tidy["note"].fillna("Not specified")
# independent flag if the note discloses a conflict between two sources
# (very useful for data-quality analysis)
tidy["has_conflict"] = tidy["note"].str.contains("CONFLICT", case=False, na=False)

# ---------------------------------------------------------------
# 7) Column: source_url
# ---------------------------------------------------------------
tidy["source_url"] = tidy["source_url"].str.strip().replace({"": np.nan})
# simple check that the URL looks valid (starts with http) to catch broken links
tidy["source_url_valid"] = tidy["source_url"].apply(
    lambda u: bool(re.match(r"^https?://", u)) if pd.notna(u) else np.nan
)

# ---------------------------------------------------------------
# 8) Final column ordering and naming
# ---------------------------------------------------------------
tidy = tidy[[
    "species", "parameter", "value_min", "value_max", "value_text", "unit",
    "is_missing", "evidence_grade", "evidence_rank", "has_conflict",
    "note", "source_url", "source_url_valid",
]].sort_values(["species", "parameter"]).reset_index(drop=True)

tidy.to_csv("plant_requirements_clean_long.csv", index=False, encoding="utf-8-sig")

# ---------------------------------------------------------------
# 9) Optional "wide" version: one row per species, each parameter as a column
#    (useful for a quick overview / comparing plants side by side)
# ---------------------------------------------------------------
wide_min = tidy.pivot_table(index="species", columns="parameter", values="value_min", aggfunc="first")
wide_max = tidy.pivot_table(index="species", columns="parameter", values="value_max", aggfunc="first")
wide_min.columns = [f"{c}__min" for c in wide_min.columns]
wide_max.columns = [f"{c}__max" for c in wide_max.columns]
wide = pd.concat([wide_min, wide_max], axis=1).sort_index(axis=1)
wide.to_csv("plant_requirements_clean_wide.csv", encoding="utf-8-sig")

print("TIDY SHAPE:", tidy.shape)
print(tidy.head(15).to_string())
print("\nMissing flags:", tidy["is_missing"].sum())
print("Conflicts flagged:", tidy["has_conflict"].sum())

TIDY SHAPE: (89, 13)
       species              parameter  value_min  value_max         value_text  unit  is_missing  evidence_grade  evidence_rank  has_conflict                                                                                                                                                                                                                                                     note                                                                                                   source_url source_url_valid
0    Aloe vera        crop_cycle_days      210.0      240.0                NaN   NaN       False        database            4.0         False                                                                                                                                Odisha SOP: first harvest 7-8 months after planting, then about 4 harvests per year; plantation profitable for 3-4 years.  https://smpbodisha.in/admin/data/ckeditor/images/060320233178133321_SO

In [2]:
print("TIDY SHAPE:", tidy.shape)

TIDY SHAPE: (89, 13)
